# First Progress Report · Evidence and reproducibility checks
This notebook reconciles model outputs, verifies holdout boundaries, and exports report-ready summaries. It does not evaluate any reserved outcome.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'config/study_design.json').exists())
sys.path.insert(0,str(ROOT/'src'))
from audit_utils import *
from baseline_utils import *
TABLES=ROOT/'outputs/tables';FIGS=ROOT/'outputs/figures'
pd.set_option('display.width',180);pd.set_option('display.max_columns',30)
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('Stage: Weeks 1-4 / development-only analyses')


Stage: Weeks 1-4 / development-only analyses


In [2]:
metrics=pd.read_csv(TABLES/'baseline_all_fold_metrics.csv')
preds=pd.read_csv(TABLES/'baseline_validation_predictions.csv',dtype={'row_id':str,'group_id':str})
recomputed=[]
for (dataset,fold,model),z in preds.groupby(['dataset','fold','model']):
    calc=score(z.y.to_numpy(),z.p.to_numpy(),rankable=model=='Logistic Regression')
    stored=metrics.loc[(metrics.dataset==dataset)&(metrics.fold.astype(str)==str(fold))&(metrics.model==model)].iloc[0]
    for m in ['roc_auc','average_precision','accuracy','brier','retention_error_pp']:
        assert np.isclose(calc[m],stored[m],atol=1e-10), (dataset,fold,model,m)
    recomputed.append({'dataset':dataset,'fold':str(fold),'model':model,'metric_reconciliation':True})
print('Reconciled',len(recomputed),'model-fold results from saved predictions.')


Reconciled 28 model-fold results from saved predictions.


In [3]:
fold_ids=pd.read_csv(TABLES/'telecom_development_fold_assignments.csv',dtype={'group_id':str})
for name in ['ibm','iranian']:
    assignment=pd.read_csv(TABLES/f'{name}_split_assignments.csv',dtype={'group_id':str})
    allowed=set(assignment.loc[assignment.split=='development','row_index'].astype(str))
    used=set(preds.loc[preds.dataset==name,'row_id'])
    assert used<=allowed
    g=fold_ids.loc[fold_ids.dataset==name]
    assert g.groupby('group_id').validation_fold.nunique().max()==1
windows=pd.read_csv(TABLES/'retail_rolling_validation_plan.csv')
assert (pd.to_datetime(windows.train_latest_label_end)<=pd.to_datetime(windows.fold)).all()
assert (pd.to_datetime(windows.validation_label_end)<=pd.Timestamp('2011-09-01')).all()
checks={'all_saved_metrics_reconciled':True,'telecom_predictions_development_only':True,
        'telecom_profile_groups_do_not_cross_folds':True,'retail_training_labels_visible_before_validation':True,
        'retail_validation_labels_end_before_final_test_cutoff':True,'final_test_evaluated':False}
(ROOT/'outputs/weeks1_4_validation_checks.json').write_text(json.dumps(checks,indent=2))
print(json.dumps(checks,indent=2))


{
  "all_saved_metrics_reconciled": true,
  "telecom_predictions_development_only": true,
  "telecom_profile_groups_do_not_cross_folds": true,
  "retail_training_labels_visible_before_validation": true,
  "retail_validation_labels_end_before_final_test_cutoff": true,
  "final_test_evaluated": false
}


In [4]:
summary=pd.read_csv(TABLES/'baseline_summary.csv')
small=summary[['dataset','model','folds','roc_auc_mean','roc_auc_sd','average_precision_mean','accuracy_mean','recall_05_mean','brier_mean']].copy()
small.to_csv(TABLES/'first_progress_report_metrics.csv',index=False)
print(small.round(3).to_string(index=False))
print('Report scope: data preparation + development exploration + baseline validation, not final deployment readiness.')


dataset               model  folds  roc_auc_mean  roc_auc_sd  average_precision_mean  accuracy_mean  recall_05_mean  brier_mean
    ibm Logistic Regression      5         0.843       0.006                   0.650          0.802           0.544       0.137
    ibm    Prior / majority      5         0.500       0.000                   0.265          0.735           0.000       0.195
iranian Logistic Regression      5         0.929       0.006                   0.755          0.900           0.436       0.071
iranian    Prior / majority      5         0.500       0.000                   0.157          0.843           0.000       0.133
 retail Logistic Regression      4         0.743       0.016                   0.708          0.669           0.717       0.213
 retail    Prior / majority      4         0.500       0.000                   0.519          0.459           0.750       0.255
Report scope: data preparation + development exploration + baseline validation, not final deployment rea

## Instructor discussion
1. Is the retail repeat-purchase definition and 90-day horizon appropriate for this project?
2. Is grouping identical Iranian candidate profiles a reasonable conservative response to missing customer IDs?
3. Should the next stage prioritize incremental predictive value, calibration, or cross-dataset explanation stability?

These are discussion points, not blockers: baseline experiments and the first progress report are complete.